# 01 · The model

Every agent starts with a model call. In this notebook you connect Strands Agents to the shared local
model, make a first call, and get a structured answer back: ShipFast's triage, which sorts a customer
message into an intent and an order ID.

**You will learn**

- how the shared model is set up in Strands, and why its settings are fixed
- how to call a model through a Strands `Agent`, and why each call here uses a fresh one
- how to get a typed, validated answer instead of free text
- where the token counts of a call are reported

**Before you start:** Ollama is running (`ollama serve`) and the shared model exists
(`ollama create qwen3-agents -f contracts/ollama/Modelfile`). See the README.

In [1]:
# Check that Ollama is running and the shared model exists. If not, this stops with the command that fixes it.
from importlib.metadata import version

from agent_contracts.harness import load_settings, verify_model

verify_model()

print("Model:         ", load_settings().model)
print("Strands Agents:", version("strands-agents"))

Model:          qwen3-agents
Strands Agents: 1.63.0


## The shared model

All three tutorials (LangGraph, CrewAI and Strands) use the same model with the same settings, so any
difference you see later comes from the framework, not the model:

| Setting | Value | Why |
|---|---|---|
| model | `qwen3-agents` | `qwen3:8b`, a small model that runs on a laptop, with the settings below built in |
| `temperature` | 0 | the most likely answer every time |
| `seed` | 42 | makes repeated runs repeatable |
| `num_ctx` | 16384 | the model's working memory in tokens; Ollama's default of 4096 is too small for agents |
| thinking | off | Qwen 3 can reason before answering; off is 8 to 10 times faster on this machine |

Strands' `OllamaModel` talks to Ollama's own API, so it can send every setting per request: the sampling
settings in `options`, and thinking off with Ollama's `think` flag through `additional_args`. One
Strands-specific point: with no model given, a Strands `Agent` uses **Amazon Bedrock**, which needs AWS
credentials. Every agent in this tutorial is given the local model explicitly.

`make_model()` lives in `strands_agents_impl/llm.py`, so these notebooks and the finished agents build the model the same way. Here is every setting it passes to `OllamaModel`:

```python
def make_model(settings: ModelSettings | None = None) -> OllamaModel:
    # The shared settings (or the ones passed in). MODEL and OLLAMA_HOST can override the first two.
    s = settings or load_settings()
    return OllamaModel(
        s.host,  # where Ollama listens, http://localhost:11434 by default
        model_id=s.model,  # "qwen3-agents" unless MODEL says otherwise
        temperature=s.temperature,  # 0: always the likeliest next token, so runs are close to repeatable
        options={"seed": s.seed, "num_ctx": s.num_ctx},  # Ollama options: seed 42, a 16,384-token context
        additional_args={"think": s.think},  # False: thinking off
    )
```

In [ ]:
# Build the model the way the agents do.
from strands_agents_impl.llm import make_model

model = make_model()  # the model used in the rest of this notebook

## A first call

Strands is agent-first: you talk to a model through an `Agent`. Calling the agent with a string runs it
and returns an `AgentResult`; `str(result)` is the reply. By default an agent also prints its reply as it
streams; `callback_handler=None` turns that off so only our own prints appear.

Ask about something the model cannot know: ShipFast is a made-up shop, and its policy is not in the
model's training data.

In [3]:
from strands import Agent


def ask(prompt: str, system: str | None = None) -> str:
    """One independent model call: a fresh agent each time, so no earlier turn leaks in."""
    agent = Agent(model=model, system_prompt=system, callback_handler=None)  # None: do not stream to the screen
    return str(agent(prompt)).strip()  # calling the agent returns an AgentResult; str() is the reply text


# Ask something the model cannot know: ShipFast is a made-up shop.
print("Model's answer:", ask("In one sentence: what is ShipFast's refund policy for lost packages?"))

Model's answer: ShipFast's refund policy for lost packages typically involves a full refund of the shipping fees once the package is confirmed lost, provided the customer submits proof of the lost package and follows the company's claim process.


The model either says it does not know or makes something up. Both are the point of the next
notebook: a model on its own only knows what it was trained on, and tools are how it reaches real data.

Why a fresh agent each time? A Strands `Agent` **remembers**: every call appends to `agent.messages`, and
the next call sees all of it. That is exactly what notebook 04 needs, and exactly what independent tests
here must avoid.

## Messages and roles

The system prompt is the agent's `system_prompt`; what you pass when calling it is the user's message.

In [4]:
# The system prompt sets the rules; the prompt is what the customer typed.
reply = ask("Hi, where is my order A1001?",
            system="You are ShipFast's support assistant. Reply in one short sentence. "
                   "You cannot look anything up yet, so never invent order details.")

print("Reply:", reply)

Reply: Your order A1001 is currently being processed and will be shipped soon.


If that reply states anything about order A1001, such as a status or a delivery date, the model
invented it: it has no access to orders. A rule in the system prompt makes invention less likely, but
it is not a guarantee, especially with a small model. That is why ShipFast checks facts with tools
(notebook 02) and checks every refund against a ledger in code, never against the model's word.

## Structured output: triage

Free text is fine for a person, but code needs fields it can check. ShipFast's first step is **triage**:
read the customer's message and return an `intent` and an `order_id`. The shape is the shared `Triage`
model from `agent_contracts`, the same class the LangGraph and CrewAI builds use.

First, look at the shape itself: the fields the model must fill in.

In [5]:
# Triage is a Pydantic model shared by all three tutorials (contracts/agent_contracts/shipfast/schema.py).
from agent_contracts.shipfast.schema import Triage

# Its JSON schema is what the model gets to see. Print each field with its allowed values and description.
for name, field in Triage.model_json_schema()["properties"].items():
    allowed = field.get("enum") or [option["type"] for option in field["anyOf"]]
    print(name)
    print("  allowed:    ", ", ".join(allowed))
    print("  description:", field["description"])

intent
  allowed:     order_status, refund_request, complaint, other
  description: order_status, refund_request, complaint, or other (out of scope)
order_id
  allowed:     string, null
  description: The order ID, such as A1001, or null if there is none


Two fields, and `intent` can only be one of four values. Pass `structured_output_model=Triage` when calling the agent and the result carries a validated `Triage`
object in `result.structured_output`. Under the hood Strands gives the model the schema as one more tool
and requires it to be called, which is why such a call uses more input tokens than a plain one. (Older
Strands versions had a separate `agent.structured_output(...)` method; it is deprecated.)

In [6]:
import warnings

# Strands asks the model provider to force a call to the schema tool. Ollama's provider cannot, so Strands
# warns on every structured call. That is expected here, so silence just that one warning.
warnings.filterwarnings("ignore", message="A ToolChoice was provided")

# The instructions for triage: what each intent means, and when order_id should be null.
TRIAGE_PROMPT = (
    "Classify the customer's latest message for ShipFast support.\n"
    "intent: order_status (where is my order, when will it arrive), refund_request (an explicit ask for a "
    "refund, return or cancellation), complaint (something is wrong, with no explicit refund ask), or other "
    "(anything that is not about orders, deliveries, refunds or complaints).\n"
    "order_id: the order ID such as A1001 if the message gives one, else null."
)


def triage(text: str) -> Triage:
    """Triage one customer message with a fresh agent, so earlier messages cannot leak in."""
    agent = Agent(model=model, system_prompt=TRIAGE_PROMPT, callback_handler=None)
    result = agent(text, structured_output_model=Triage)  # Strands adds the schema as a tool the model must call
    return result.structured_output  # the validated Triage object


result = triage("My order A1002 never arrived. I want a refund.")

print("Returned:", type(result).__name__)
print("intent:  ", result.intent)
print("order_id:", result.order_id)

Returned: Triage
intent:   refund_request
order_id: A1002


Behind the scenes, Strands asks the model provider to **force** a call to that schema tool. Ollama's
provider cannot force a tool, so Strands warns ("A ToolChoice was provided to this provider but is not
supported") and carries on; the cell above silences that one expected warning. The model chose to call
the tool anyway. The same limit came up in the LangGraph tutorial: with this setup nothing can force a
local model to call a particular tool, only instructions can.

Now run it over the opening message of every ShipFast scenario and compare with the expected answer
from `scenarios.json`. (Some scenarios have a second turn that only makes sense with the first one in
memory; that is notebook 04.)

In [7]:
# Triage the first message of every ShipFast scenario and compare it with the expected answer.
import time

from agent_contracts.shipfast.scoring import load_scenarios

seen = set()  # opening messages already tried: two scenarios start with the same one
correct = 0

print(f"{'check':6} {'time':>5}  {'intent':15} {'order':6} customer message")
for scenario in load_scenarios():
    text = scenario["turns"][0]  # the customer's first message
    if text in seen:
        continue
    seen.add(text)
    expected = scenario["expected"]  # the right intent and order ID, from scenarios.json

    start = time.perf_counter()
    got = triage(text)
    seconds = time.perf_counter() - start

    ok = (got.intent, got.order_id) == (expected["intent"], expected["order_id"])
    correct += ok
    print(f"{'ok' if ok else 'MISS':6} {seconds:4.1f}s  {got.intent:15} {str(got.order_id):6} {text}")

print(f"\n{correct} of {len(seen)} messages triaged correctly")

check   time  intent          order  customer message


ok      5.4s  order_status    A1001  Hi, where is my order A1001?


ok      5.9s  refund_request  A1002  My order A1002 never arrived and tracking says it's lost. I want a refund.


ok      6.0s  refund_request  A1003  Order A1003 is more than a week late. Please cancel it and refund me.


ok      5.2s  complaint       A1004  My vase set from order A1004 arrived broken.


ok      5.4s  refund_request  A1005  I want to return order A1005, the stand wobbles.


ok      5.2s  order_status    A9999  Where is order A9999?


ok     10.7s  other           None   Can you write me a poem about cats?

7 of 7 messages triaged correctly


## What a call costs

Every `AgentResult` carries metrics, including the tokens Ollama reported, summed over the agent's model
calls. Notebook 07 collects these into the shared harness so the three frameworks are measured the same
way.

In [8]:
# A plain agent call (no structured output), to read the metrics on its AgentResult.
agent = Agent(model=model, system_prompt=TRIAGE_PROMPT, callback_handler=None)
result = agent("Where is order A9999?")
usage = result.metrics.accumulated_usage  # tokens, summed over the agent's model calls

print("Reply:        ", " ".join(str(result).split()))  # the model answered in JSON; shown on one line
print("Input tokens: ", usage["inputTokens"], "(instructions and message)")
print("Output tokens:", usage["outputTokens"], "(the model's answer)")
print("Model calls:  ", result.metrics.cycle_count)

Reply:         { "intent": "order_status", "order_id": "A9999" }
Input tokens:  121 (instructions and message)
Output tokens: 23 (the model's answer)
Model calls:   1


## How the others do this

- **LangGraph** (through LangChain) uses `ChatOllama`, also on Ollama's own API, and calls the model
  directly with `llm.invoke(...)`: no agent needed for a single call. Structured output is
  `llm.with_structured_output(Triage)`.
- **CrewAI** uses `LLM("ollama/qwen3-agents")`, which reaches Ollama through its OpenAI-compatible API;
  `llm.call(messages, response_model=Triage)` returns the structured result.

All three get the same `Triage` class from `agent_contracts`.

## Takeaways

- In Strands you call the model through an `Agent`; give it the local model explicitly, or it uses Bedrock.
- An agent keeps its conversation, so independent calls use a fresh agent.
- `structured_output_model` turns a reply into a validated object your code can trust.
- A model alone cannot see ShipFast's orders or policy. Next: **02 · Tools**.